In [ ]:
import pandas as pd
import numpy as np

# Convert purchase timestamp
dataset['order_purchase_timestamp'] = pd.to_datetime(dataset['order_purchase_timestamp'], errors='coerce')
dataset['Order_Purchase_Time'] = dataset['order_purchase_timestamp'].dt.time

# Convert 'order estimated delivery date' from DateTime to Date

dataset['order_estimated_delivery_date'] = pd.to_datetime(dataset['order_estimated_delivery_date'], errors='coerce')
dataset['order_estimated_delivery_date'] = dataset['order_estimated_delivery_date'].dt.date

# Convert delivered customer date
dataset['Order_Delivered_Customer_Date'] = pd.to_datetime(dataset['order_delivered_customer_date'], errors='coerce')
dataset['Order_Delivered_Customer_Time'] = dataset['Order_Delivered_Customer_Date'].dt.time

# Convert approved date
dataset['Order_Approved_Date'] = pd.to_datetime(dataset['order_approved_at'], errors='coerce')
dataset['Order_Approved_Time'] = dataset['Order_Approved_Date'].dt.time

# Convert delivered carrier date
dataset['Order_Delivered_Carrier_Date'] = pd.to_datetime(dataset['order_delivered_carrier_date'], errors='coerce')
dataset['Order_Delivered_Carrier_Time'] = dataset['Order_Delivered_Carrier_Date'].dt.time

# Create Year and Month
dataset['Year_of_purchase'] = dataset['order_purchase_timestamp'].dt.year
dataset['Month_of_purchase'] = dataset['order_purchase_timestamp'].dt.strftime('%b')

# Filling Missing Actual Delivery Dates and Times with corresponding Delivered Carrier Date and Times
dataset['Order_Delivered_Customer_Date'] = dataset['Order_Delivered_Customer_Date'].fillna(dataset['Order_Delivered_Carrier_Date'])
dataset['Order_Delivered_Customer_Time'] = dataset['Order_Delivered_Customer_Time'].fillna(dataset['Order_Delivered_Carrier_Time'])

# Num Of Delay Days


dataset['Delay_Days'] = (
    (dataset['Order_Delivered_Customer_Date'] - pd.to_datetime(dataset['order_estimated_delivery_date']))
    .dt.days.fillna(0).astype(int)
)



dataset['Delay_Days'] = dataset['Delay_Days'].clip(lower=0)


# Is_late Flag

dataset['Is_late'] = np.where(

    dataset['Delay_Days']>0,1,0
)



# Dropping Unneccary Columns

dataset.drop(columns=['order_purchase_timestamp','order_delivered_customer_date','order_approved_at','order_delivered_carrier_date'], inplace=True)


dataset['order_status'] = dataset['order_status'].str.title()

# Bucketing Revenue Status according to Order status

conditions = [
    (dataset['order_status']=='Delivered'),
    (dataset['order_status'].isin(['Canceled','Unavailable'])),
    (dataset['order_status'].isin(['Created','Shipped','Invoiced','Approved','Processing']))
]

choices = [
    'Actual Revenue',
    'Canceled/Refunded',
    'Pending/In-Transit'
]

dataset['Revenue Status'] = np.select(conditions, choices, default='Other')


# Dropping Only 1 order that has no Customer/Carrier Delivery date, impact is negligable to our dataset

dataset = dataset.drop(
    dataset[(dataset['order_status'] == 'Delivered') & 
            (dataset['Order_Delivered_Customer_Date'].isna())].index
)


# String Standartdization

dataset.columns = dataset.columns.str.replace('_', ' ').str.title()

output = dataset
